# Desafio 1 — Bronze: Frentes Parlamentares

## Objetivo
Coletar dados brutos da API de Dados Abertos da Câmara sobre frentes parlamentares
e seus membros, armazenando na camada bronze (dados imutáveis com metadados de auditoria).

## Origem dos dados
- API: https://dadosabertos.camara.leg.br/swagger/api.html
- Endpoints utilizados:
  - `/frentes` — lista de frentes parlamentares
  - `/frentes/{id}/membros` — membros de cada frente

## Estratégia
- Paginação completa: percorrer todas as páginas da API até esgotar os resultados
- Coleta dos membros: uma chamada por frente (1.440 frentes)
- Pausa de 0,3 segundos entre chamadas para respeitar a API
- Campos de auditoria: data_coleta, origem


In [0]:
# ============================================================
# CAMADA BRONZE - FRENTES PARLAMENTARES
# ============================================================
# Objetivo: Coletar dados BRUTOS da API e salvar como JSON
# Esses dados NUNCA serão alterados (imutáveis)
# ============================================================

import requests
import json
import time
from datetime import datetime

# ============================================================
# FUNÇÃO 1: Coletar TODAS as frentes (paginação completa)
# ============================================================
def coletar_todas_frentes(itens_por_pagina=100):
    """
    Percorre TODAS as páginas da API de frentes e retorna uma lista completa.
    
    Parâmetros:
        itens_por_pagina: quantos registros por página (máx 100)
    
    Retorna:
        Lista com todas as frentes (lista de dicionários)
    """
    url = "https://dadosabertos.camara.leg.br/api/v2/frentes"
    todas_frentes = []
    pagina_atual = 1
    
    print(f"Iniciando coleta de frentes...")
    
    while True:
        # Configurar parâmetros da página atual
        params = {
            "itens": itens_por_pagina,
            "pagina": pagina_atual
        }
        
        # Fazer requisição
        response = requests.get(url, params=params)
        
        # Verificar se funcionou
        if response.status_code != 200:
            print(f"Erro na página {pagina_atual}: {response.status_code}")
            print(response.text)
            break
        
        # Extrair dados
        dados = response.json()
        frentes_pagina = dados['dados']
        
        # Se não veio nada, chegamos ao fim
        if not frentes_pagina:
            print(f"Página {pagina_atual} vazia - fim da coleta")
            break
        
        # Adicionar à lista total
        todas_frentes.extend(frentes_pagina)
        
        print(f"Página {pagina_atual}: {len(frentes_pagina)} frentes coletadas | Total acumulado: {len(todas_frentes)}")
        
        # Verificar se é a última página
        # (quando o link "next" não existe ou a página atual é a última)
        links = dados.get('links', [])
        tem_proxima = any(link['rel'] == 'next' for link in links)
        
        if not tem_proxima:
            print("Última página alcançada!")
            break
        
        # Ir para próxima página
        pagina_atual += 1
        
        # PAUSA EDUCADA: pequeno delay para não sobrecarregar a API
        time.sleep(0.5)
    
    print(f"\nColeta finalizada! Total de frentes: {len(todas_frentes)}")
    return todas_frentes

# ============================================================
# FUNÇÃO 2: Coletar membros de UMA frente específica
# ============================================================
def coletar_membros_frente(id_frente):
    """
    Coleta os membros de uma frente parlamentar específica.
    
    Parâmetros:
        id_frente: ID da frente (ex: 55688)
    
    Retorna:
        Lista de membros (cada membro é um dicionário)
    """
    url = f"https://dadosabertos.camara.leg.br/api/v2/frentes/{id_frente}/membros"
    
    response = requests.get(url)
    
    if response.status_code == 200:
        dados = response.json()
        membros = dados.get('dados', [])
        return membros
    else:
        print(f"Erro ao coletar membros da frente {id_frente}: {response.status_code}")
        return []

# ============================================================
# TESTE RÁPIDO DAS FUNÇÕES
# ============================================================

# Testar com apenas 2 páginas (depois mudamos para TODAS)
print("=" * 60)
print("TESTE: Coletando frentes (apenas 2 páginas)")
print("=" * 60)

url = "https://dadosabertos.camara.leg.br/api/v2/frentes"
frentes_teste = []

for pagina in [1, 2]:
    params = {"itens": 100, "pagina": pagina}
    response = requests.get(url, params=params)
    if response.status_code == 200:
        dados = response.json()
        frentes_teste.extend(dados['dados'])
        print(f"Página {pagina}: {len(dados['dados'])} frentes")

print(f"\nTotal coletado (teste): {len(frentes_teste)}")

# Testar membros da primeira frente
if frentes_teste:
    primeira_frente = frentes_teste[0]
    print(f"\nPrimeira frente: {primeira_frente['titulo'][:80]}...")
    print(f"ID: {primeira_frente['id']}")
    
    membros = coletar_membros_frente(primeira_frente['id'])
    print(f"Membros encontrados: {len(membros)}")
    
    # Mostrar os 3 primeiros membros
    for m in membros[:3]:
        print(f"  - {m['nome']} ({m['siglaPartido']}-{m['siglaUf']}) [{m['titulo']}]")

In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS bronze;

In [0]:
# ============================================================
# PREPARAR DADOS DAS FRENTES PARA INSERÇÃO
# ============================================================
import json
from datetime import datetime

data_coleta = datetime.now().isoformat()

# Criar uma lista de tuplas (cada tupla = uma linha da tabela)
# Ordem das colunas: id_frente, titulo, uri, id_legislatura, dado_bruto_json, data_coleta, origem
dados_para_inserir = []

for frente in frentes_teste:  # frentes_teste tem 200 frentes do teste anterior
    linha = (
        frente['id'],                          # id_frente
        frente['titulo'],                      # titulo
        frente['uri'],                         # uri
        frente['idLegislatura'],               # id_legislatura
        json.dumps(frente, ensure_ascii=False), # dado_bruto_json (JSON completo)
        data_coleta,                           # data_coleta
        'api_camara_frentes'                   # origem
    )
    dados_para_inserir.append(linha)

print(f"Registros preparados: {len(dados_para_inserir)}")
print(f"\nExemplo da primeira linha:")
print(f"  ID: {dados_para_inserir[0][0]}")
print(f"  Título: {dados_para_inserir[0][1][:80]}...")
print(f"  Legislatura: {dados_para_inserir[0][3]}")

# Desafio 3: Criar Tabela Bronze

In [0]:
# ============================================================
# CRIAR TABELA BRONZE E INSERIR DADOS
# ============================================================

# Converter para DataFrame Spark com schema explícito
from pyspark.sql.types import StructType, StructField, LongType, StringType, IntegerType

schema = StructType([
    StructField("id_frente", LongType(), True),
    StructField("titulo", StringType(), True),
    StructField("uri", StringType(), True),
    StructField("id_legislatura", IntegerType(), True),
    StructField("dado_bruto_json", StringType(), True),
    StructField("data_coleta", StringType(), True),
    StructField("origem", StringType(), True)
])

df = spark.createDataFrame(dados_para_inserir, schema=schema)

# Criar a tabela Delta diretamente do DataFrame
df.write \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("bronze.frentes")

print(f"Tabela bronze.frentes criada com {df.count()} registros!")

In [0]:
%sql
SELECT 
    COUNT(*) AS total, 
    COUNT(DISTINCT id_frente) AS frentes_unicas,
    MIN(id_legislatura) AS min_leg,
    MAX(id_legislatura) AS max_leg
FROM bronze.frentes;

In [0]:
# ============================================================
# COLETA COMPLETA: TODAS AS 1.440 FRENTES
# ============================================================
import time


print("Iniciando coleta COMPLETA de todas as frentes...")
print(" Isso pode levar alguns minutos (1440 frentes, ~15 páginas)\n")

frentes_completas = coletar_todas_frentes(itens_por_pagina=100)

print(f"\n✅ Coleta finalizada!")
print(f"Total de frentes coletadas: {len(frentes_completas)}")
print(f"Exemplo: {frentes_completas[0]['titulo'][:80]}...")

In [0]:
# Preparar os 1.440 registros com metadados de auditoria
from datetime import datetime
import json

data_coleta = datetime.now().isoformat()
dados_completos = []

for frente in frentes_completas:
    linha = (
        frente['id'],
        frente['titulo'],
        frente['uri'],
        frente['idLegislatura'],
        json.dumps(frente, ensure_ascii=False),
        data_coleta,
        'api_camara_frentes'
    )
    dados_completos.append(linha)

print(f"Registros preparados: {len(dados_completos)}")

In [0]:
%sql
-- Limpar a tabela antiga (200 registros de teste)
DROP TABLE IF EXISTS bronze.frentes;

In [0]:
# Criar schema tipado e salvar
from pyspark.sql.types import StructType, StructField, LongType, StringType, IntegerType

schema = StructType([
    StructField("id_frente", LongType(), True),
    StructField("titulo", StringType(), True),
    StructField("uri", StringType(), True),
    StructField("id_legislatura", IntegerType(), True),
    StructField("dado_bruto_json", StringType(), True),
    StructField("data_coleta", StringType(), True),
    StructField("origem", StringType(), True)
])

df = spark.createDataFrame(dados_completos, schema=schema)

df.write \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("bronze.frentes")

print(f"Tabela bronze.frentes atualizada com {df.count()} registros.")

In [0]:
%sql
-- Verificar
SELECT 
    COUNT(*) AS total,
    COUNT(DISTINCT id_frente) AS frentes_unicas,
    MIN(id_legislatura) AS min_leg,
    MAX(id_legislatura) AS max_leg
FROM bronze.frentes;

In [0]:
# Recarregar frentes da tabela bronze
df_bronze = spark.sql("SELECT * FROM bronze.frentes")
frentes_completas = df_bronze.collect()  # Lista de objetos Row

# Converter para lista de dicionarios
frentes_completas = [row.asDict() for row in frentes_completas]

print(f"Frentes carregadas da tabela bronze: {len(frentes_completas)}")

In [0]:
# Recarregar frentes da tabela bronze e inspecionar formato
df_bronze = spark.sql("SELECT * FROM bronze.frentes")

# Ver nomes das colunas
print("Colunas da tabela bronze.frentes:")
print(df_bronze.columns)

# Pegar uma linha de exemplo e inspecionar
exemplo = df_bronze.limit(1).collect()[0]
print("\nTipo do objeto:", type(exemplo))
print("Conteudo:", exemplo)

# Converter para dicionario usando os nomes corretos das colunas
frentes_completas = []
for row in df_bronze.collect():
    frente_dict = {
        'id': row['id_frente'],           
        'titulo': row['titulo'],
        'uri': row['uri'],
        'idLegislatura': row['id_legislatura'], 
        'dado_bruto_json': row['dado_bruto_json'],
        'data_coleta': row['data_coleta'],
        'origem': row['origem']
    }
    frentes_completas.append(frente_dict)

print(f"\nFrentes carregadas: {len(frentes_completas)}")
print(f"Exemplo da primeira frente (id): {frentes_completas[0]['id']}")

In [0]:
# Coletar membros de todas as frentes
import requests
import time
import time

membros_por_frente = {}  # Dicionario: chave = id_frente, valor = lista de membros
erros_coleta = []        # Lista para registrar frentes que falharam

total_frentes = len(frentes_completas)
print(f"Iniciando coleta de membros para {total_frentes} frentes...")
print("(Uma pequena pausa a cada chamada para respeitar a API)\n")

for i, frente in enumerate(frentes_completas, start=1):
    id_frente = frente['id']
    
    # Mostrar progresso a cada 100 frentes
    if i % 100 == 0 or i == 1 or i == total_frentes:
        print(f"Processando frente {i}/{total_frentes} (ID: {id_frente})")
    
    # Chamar o endpoint de membros
    url = f"https://dadosabertos.camara.leg.br/api/v2/frentes/{id_frente}/membros"
    response = requests.get(url)
    
    if response.status_code == 200:
        dados = response.json()
        membros = dados.get('dados', [])
        membros_por_frente[id_frente] = membros
    else:
        erros_coleta.append({
            'id_frente': id_frente,
            'status': response.status_code,
            'mensagem': response.text[:200]
        })
    
    # Pausa para respeitar a API (0.3 segundos entre chamadas)
    time.sleep(0.3)

print(f"\nColeta finalizada!")
print(f"Frentes com membros coletados: {len(membros_por_frente)}")
print(f"Frentes com erro: {len(erros_coleta)}")

if erros_coleta:
    print("\nExemplos de erros:")
    for erro in erros_coleta[:5]:
        print(f"  ID {erro['id_frente']}: Status {erro['status']}")

In [0]:
# Verificar estatisticas da coleta de membros
total_frentes_com_membros = len(membros_por_frente)
total_membros = sum(len(lista) for lista in membros_por_frente.values())

print(f"Frentes que retornaram membros: {total_frentes_com_membros}")
print(f"Frentes com erro na coleta: {len(erros_coleta)}")
print(f"Total de registros de membros: {total_membros}")
print(f"Media de membros por frente: {total_membros / max(total_frentes_com_membros, 1):.1f}")

# Ver um exemplo
if membros_por_frente:
    primeiro_id = list(membros_por_frente.keys())[0]
    print(f"\nExemplo - Frente {primeiro_id}:")
    for m in membros_por_frente[primeiro_id][:3]:
        print(f"  {m['nome']} ({m['siglaPartido']}-{m['siglaUf']}) - {m['titulo']}")

In [0]:
# Transformar dicionario de membros em lista plana para tabela
import json
from datetime import datetime

data_coleta = datetime.now().isoformat()
membros_flat = []

for id_frente, lista_membros in membros_por_frente.items():
    for membro in lista_membros:
        linha = (
            id_frente,                                 # id_frente
            membro.get('id'),                          # id_deputado
            membro.get('nome'),                        # nome
            membro.get('siglaPartido'),                # sigla_partido
            membro.get('siglaUf'),                     # sigla_uf
            membro.get('titulo'),                      # cargo (Coordenador, Membro, etc.)
            membro.get('codTitulo'),                   # codigo do cargo
            membro.get('dataInicio'),                  # data_inicio
            membro.get('dataFim'),                     # data_fim
            json.dumps(membro, ensure_ascii=False),    # dado_bruto_json
            data_coleta,                               # data_coleta
            'api_camara_frentes_membros'               # origem
        )
        membros_flat.append(linha)

print(f"Registros preparados para insercao: {len(membros_flat)}")
print(f"Exemplo (primeira linha): ID Frente={membros_flat[0][0]}, Deputado={membros_flat[0][2]}, Cargo={membros_flat[0][5]}")

In [0]:
%sql
DROP TABLE IF EXISTS bronze.frentes_membros;

CREATE TABLE IF NOT EXISTS bronze.frentes_membros (
    id_frente BIGINT,
    id_deputado BIGINT,
    nome STRING,
    sigla_partido STRING,
    sigla_uf STRING,
    cargo STRING,
    cod_cargo INT,
    data_inicio STRING,
    data_fim STRING,
    dado_bruto_json STRING,
    data_coleta STRING,
    origem STRING
)
USING DELTA
COMMENT 'Camada Bronze - Membros das Frentes Parlamentares (dados brutos da API)';

In [0]:
from pyspark.sql.types import StructType, StructField, LongType, StringType, IntegerType

schema = StructType([
    StructField("id_frente", LongType(), True),
    StructField("id_deputado", LongType(), True),
    StructField("nome", StringType(), True),
    StructField("sigla_partido", StringType(), True),
    StructField("sigla_uf", StringType(), True),
    StructField("cargo", StringType(), True),
    StructField("cod_cargo", IntegerType(), True),
    StructField("data_inicio", StringType(), True),
    StructField("data_fim", StringType(), True),
    StructField("dado_bruto_json", StringType(), True),
    StructField("data_coleta", StringType(), True),
    StructField("origem", StringType(), True)
])

df_membros = spark.createDataFrame(membros_flat, schema=schema)

df_membros.write \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("bronze.frentes_membros")

print(f"Tabela bronze.frentes_membros criada com {df_membros.count()} registros.")

In [0]:
%sql
SELECT 
    COUNT(*) AS total_registros,
    COUNT(DISTINCT id_frente) AS frentes_unicas,
    COUNT(DISTINCT id_deputado) AS deputados_unicos,
    COUNT(DISTINCT sigla_partido) AS partidos_unicos,
    COUNT(DISTINCT sigla_uf) AS ufs_unicas
FROM bronze.frentes_membros;

In [0]:
%sql
SELECT 
    distinct data_fim FROM bronze.frentes_membros